In [1]:
!pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic

  Using cached mcp-1.28.1-py3-none-any.whl.metadata (9.4 kB)
Using cached mcp-1.28.1-py3-none-any.whl (222 kB)
  Attempting uninstall: mcp
    Found existing installation: mcp 2.3.0
    Uninstalling mcp-2.3.0:
      Successfully uninstalled mcp-2.3.0


In [2]:
!pip install "crewai[azure-ai-inference]"
!pip install litellm

In [1]:
import os
from datetime import datetime, timezone
import requests
from crewai.llm import LLM
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool

In [2]:
AZURE_OPENAI_API_KEY = "2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"
AZURE_OPENAI_ENDPOINT = "https://openai-api-management-gw.azure-api.net/"
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"
AZURE_OPENAI_DEPLOYMENT = "gpt-5-mini"

In [3]:
# Tavily API Key
TAVILY_API_KEY = "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"#"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"

In [4]:
# 1. Test Tavily API Connection
try:
    print("Testing Tavily API...")
    test_search = search_tool._run("AI healthcare advancements 2024")
    print("Tavily API is working perfectly!")
    print("Sample Output:\n", "\n".join(test_search.split("\n")[:2]))
except Exception as e:
    print(f"Tavily API Error: {e}")

Testing Tavily API...
Tavily API Error: name 'search_tool' is not defined


In [5]:
# ---- Custom CrewAI Tool for Web Search ----
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
        results = data.get("results", [])

        if not results:
            return "No search results returned."

        formatted_results = []
        for r in results:
            formatted_results.append(f"{r.get('title', 'Untitled')} - {r.get('url', '')}")

        return "\n".join(formatted_results)


search_tool = TavilySearchTool()


In [6]:
# llm = LLM(
#     model="azure/gpt-5-mini",
#     api_key="2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC",
#     base_url="https://openai-api-management-gw.azure-api.net/",
#     api_version="2024-12-01-preview",
#     is_litellm=True,
#     temperature=0,
#     max_completion_tokens=3500
# )
# --------------------------------------------------
# Azure GPT-5-mini LLM via CrewAI LiteLLM path
# --------------------------------------------------
llm = LLM(
    model="azure/gpt-5-mini",
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    is_litellm=True,
    temperature=0,
    max_completion_tokens=1500
)

In [7]:
# 2. Test Azure OpenAI Connection
try:
    print("Testing Azure OpenAI connection...")
    test_response = llm.call(messages=[{"role": "user", "content": "Respond with 'Azure OpenAI connection successful!'"}])
    print(test_response)
except Exception as e:
    print(f"Azure OpenAI Error: {e}")

Testing Azure OpenAI connection...
Azure OpenAI connection successful!


In [8]:
# # ---- Agents ----
# researcher = Agent(
#     role="AI Researcher",
#     goal="Find the latest advancements in AI for FMCG using web search",
#     backstory="Expert in AI research trends who gathers real-time information from the web.",
#     verbose=True,
#      allow_delegation=False,
#     llm=llm,

#     max_iter=1,
#     tools=[search_tool]
# )
# writer = Agent(
#     role="Technical Writer",
#     goal="Summarize research into an executive report",
#     backstory="Expert in executive summaries.",
#     verbose=True,
#     allow_delegation=False,
#     llm=llm,
#     tools=[search_tool]
# )

# --------------------------------------------------
# Agents for serial and parallel execution  exercise 2
# --------------------------------------------------
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for healthcare",
    backstory=(
        "You are an expert in artificial intelligence and stay updated "
        "with the latest research trends in healthcare."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    tools=[search_tool]
)

writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory=(
        "You are an experienced technical writer with expertise in "
        "summarizing healthcare research for executives."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm
)


In [9]:
import asyncio
# import crewai process for parallel execution
from crewai import Process

print('CrewAI version check:')
try:
    import crewai
    print(crewai.__version__)
except Exception as e:
    print('crewai import issue:', e)

print('MCP version check:')
try:
    import mcp
    print(mcp.__version__)
except Exception as e:
    print('mcp import issue:', e)

print('LiteLLM version check:')
try:
    import litellm
    print(litellm.__version__)
except Exception as e:
    print('litellm import issue:', e)

print('Loaded library compatibility check complete.')


CrewAI version check:
1.15.28
MCP version check:
mcp import issue: module 'mcp' has no attribute '__version__'
LiteLLM version check:
litellm import issue: module 'litellm' has no attribute '__version__'
Loaded library compatibility check complete.


In [10]:
%pip install "litellm==1.105.1"

Note: you may need to restart the kernel to use updated packages.


In [11]:
# # ---- Tasks ----
# task_research = Task(
#     description="Search the web and identify the top 3 recent advancements in AI for FMCG.",
#     expected_output="Detailed notes explaining three recent AI advancements in FMCG with examples.",
#     agent=researcher
# )

# task_write = Task(
#     description="""
# Write a concise executive summary using the research notes.

# Requirements:
# - Maximum 100 words
# - Use bullet points
# - Focus only on the 3 key advancements
# """,
#     expected_output="Executive summary of AI advancements in FMCG.",
#     agent=writer,
#     context=[task_research]
# )


# --------------------------------------------------
# SERIAL EXECUTION
# --------------------------------------------------
task_research = Task(
    description=(
        "Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4 sentences."
        "Do not call tools again after getting results."
    ),
    expected_output=(
        "Detailed notes on three advancements, with names and explanations."
    ),
    agent=researcher
)

task_write = Task(
    description=(
        "Write a short executive summary using the research notes provided by the AI Researcher. "
        "Limit the answer to about 100 words."
    ),
    expected_output=(
        "An executive summary report of the top 3 AI advancements in healthcare."
    ),
    agent=writer,
    # The context parameter explicitly instructs task_write to wait for and consume the output of task_research.
    # This enforces a linear dependency chain when the crew is run in sequential mode.
    context=[task_research]
)



crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    # agents=[writer],
    # tasks=[task_write],
    process=Process.sequential,
    verbose=True
)


serial_result = await crew_serial.kickoff_async()
serial_end = datetime.now(timezone.utc)


print("\n[Serial Result]:\n")
try:
    print(serial_result.raw)
except AttributeError:
    print(serial_result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 33f7ed9d-1635-467c-8227-2bccfa1dd257                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4          │
│  sentences.Do not call tools again after getting results.                                                       │
│  ID: da68e4bd-ed2e-429a-921b-f574ab419a40                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4          │
│  sentences.Do not call tools again after getting results.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': "recent advancements AI healthcare 2024 2025 top advancements large language models            │
│  multimodal foundation models AI drug discovery AI medical imaging federated learning personalized medicine...  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search executed with result: Large language models for clinical artificial intelligence in healthcare a systematic review | Discover Artificial Intelligence | Springer Nature Link - https://link.springer.com/article/10.1007/s4416...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Executive summary — Top 3 AI advancements in healthcare                                                        │
│                                                                                                                 │
│  1) Multimodal foundation models & clinical LLMs: Combined text, imaging, and EHR inputs enable automated       │
│  radiology/pathology interpretation, guideline‑aware decision support, and concise clinical summaries. They     │
│  reduce model sprawl and improve diagnostic reasoning but require domain tuning, calibration, and regulatory    │
│  validation.                                                                                                    │
│                                                                                                                 │
│  2) Generative AI + structural biology for drug discovery: AlphaFold‑enabled structure awareness plus de‑novo   │
│  generative chemistry and closed‑loop synthesis/assay feedback shorten hit‑to‑lead cycles and increase          │
│  candidate novelty; translational reproducibility and regulatory scrutiny remain key barriers.                  │
│                                                                                                                 │
│  3) AI‑driven multi‑omics & spatial biology: Integrated single‑cell, proteomic and spatial analyses yield       │
│  finer patient stratification, predictive biomarkers, and therapy matching; standardization, cross‑site         │
│  validation, and privacy safeguards are essential for clinical adoption.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short executive summary using the research notes provided by the AI Researcher. Limit the        │
│  answer to about 100 words.                                                                                     │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[Serial Result]:

Executive summary — Top 3 AI advancements in healthcare

1) Multimodal foundation models & clinical LLMs: Combined text, imaging, and EHR inputs enable automated radiology/pathology interpretation, guideline‑aware decision support, and concise clinical summaries. They reduce model sprawl and improve diagnostic reasoning but require domain tuning, calibration, and regulatory validation.

2) Generative AI + structural biology for drug discovery: AlphaFold‑enabled structure awareness plus de‑novo generative chemistry and closed‑loop synthesis/assay feedback shorten hit‑to‑lead cycles and increase candidate novelty; translational reproducibility and regulatory scrutiny remain key barriers.

3) AI‑driven multi‑omics & spatial biology: Integrated single‑cell, proteomic and spatial analyses yield finer patient stratification, predictive biomarkers, and therapy matching; standardization, cross‑site validation, and privacy safeguards are essential for clinical adoption.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 33f7ed9d-1635-467c-8227-2bccfa1dd257                                                                       │
│  Final Output: Executive summary — Top 3 AI advancements in healthcare                                          │
│                                                                                                                 │
│  1) Multimodal foundation models & clinical LLMs: Combined text, imaging, and EHR inputs enable automated       │
│  radiology/pathology interpretation, guideline‑aware decision support, and concise clinical summaries. They     │
│  reduce model sprawl and improve diagnostic reasoning but require domain tuning, calibration, and regulatory    │
│  validation.                                                                                                    │
│                                                                                                                 │
│  2) Generative AI + structural biology for drug discovery: AlphaFold‑enabled structure awareness plus de‑novo   │
│  generative chemistry and closed‑loop synthesis/assay feedback shorten hit‑to‑lead cycles and increase          │
│  candidate novelty; translational reproducibility and regulatory scrutiny remain key barriers.                  │
│                                                                                                                 │
│  3) AI‑driven multi‑omics & spatial biology: Integrated single‑cell, proteomic and spatial analyses yield       │
│  finer patient stratification, predictive biomarkers, and therapy matching; standardization, cross‑site         │
│  validation, and privacy safeguards are essential for clinical adoption.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
import importlib.metadata as md

for p in ["crewai", "crewai-tools", "litellm", "mcp", "tavily-python"]:
    try:
        print(p, md.version(p))
    except md.PackageNotFoundError:
        print(p, "not installed")

crewai 1.15.28
crewai-tools not installed
litellm 1.105.1
mcp 1.28.1
tavily-python 0.8.5


In [16]:
# --------------------------------------------------
# PARALLEL EXECUTION
# --------------------------------------------------
task_parallel_1 = Task(
    description=(
        "Use web search to list 5 AI companies focusing on drug discovery. "
        "For each company, give one short line about what they specialize in."
    ),
    expected_output="Company names and what they specialize in.",
    async_execution=True,
    agent=researcher
)

task_parallel_2 = Task(
    description=(
        "Write a short report on how AI is transforming patient diagnostics. "
        "Limit the answer to about 100 words."
    ),
    expected_output="A short report with examples and explanation.",
    
    agent=writer
)

print(f"\n[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S %Z')}] Starting parallel execution")

crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2],
    verbose=True
)

parallel_start = datetime.now(timezone.utc)
print(f"[{parallel_start.strftime('%Y-%m-%d %H:%M:%S %Z')}] Parallel kickoff started")
parallel_result = await crew_parallel.kickoff_async()
parallel_end = datetime.now(timezone.utc)
print(f"[{parallel_end.strftime('%Y-%m-%d %H:%M:%S %Z')}] Parallel kickoff completed in {(parallel_end - parallel_start).total_seconds():.2f}s")

print("\n[Parallel Result]:\n")
try:
    print(parallel_result.raw)
except AttributeError:
    print(parallel_result)



[2026-10-19 18:56:23 UTC] Starting parallel execution
[2026-10-19 18:56:23 UTC] Parallel kickoff started


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e68034f8-0030-4da2-a6da-0c6dbb08b666                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│  ID: b4b825db-d5f0-40ee-a140-9eab282b4c22                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#8) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'Insilico Medicine what they specialize in AI drug discovery company description 2024'}        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  AI is reshaping patient diagnostics by accelerating biomarker discovery, refining imaging interpretation, and  │
│  enabling precision stratification. Examples: Insilico Medicine’s generative models reveal novel targets and    │
│  biomarker signatures; Atomwise’s AtomNet speeds discovery of ligand probes useful in molecular assays;         │
│  Exscientia automates design of diagnostic reagents and candidate molecules more rapidly; BenevolentAI’s        │
│  knowledge graph and causal-AI uncover mechanistic hypotheses and repurposing opportunities that inform         │
│  diagnostic markers; Recursion combines high‑throughput cellular imaging with machine learning to detect        │
│  disease phenotypes and predictive signatures. Together these advances yield earlier detection, more specific   │
│  biomarkers, and personalized diagnostic pathways that improve clinical decision-making.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short report on how AI is transforming patient diagnostics. Limit the answer to about 100        │
│  words.                                                                                                         │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[2026-10-19 18:56:49 UTC] Parallel kickoff completed in 25.63s

[Parallel Result]:

AI is reshaping patient diagnostics by accelerating biomarker discovery, refining imaging interpretation, and enabling precision stratification. Examples: Insilico Medicine’s generative models reveal novel targets and biomarker signatures; Atomwise’s AtomNet speeds discovery of ligand probes useful in molecular assays; Exscientia automates design of diagnostic reagents and candidate molecules more rapidly; BenevolentAI’s knowledge graph and causal-AI uncover mechanistic hypotheses and repurposing opportunities that inform diagnostic markers; Recursion combines high‑throughput cellular imaging with machine learning to detect disease phenotypes and predictive signatures. Together these advances yield earlier detection, more specific biomarkers, and personalized diagnostic pathways that improve clinical decision-making.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e68034f8-0030-4da2-a6da-0c6dbb08b666                                                                       │
│  Final Output: AI is reshaping patient diagnostics by accelerating biomarker discovery, refining imaging        │
│  interpretation, and enabling precision stratification. Examples: Insilico Medicine’s generative models reveal  │
│  novel targets and biomarker signatures; Atomwise’s AtomNet speeds discovery of ligand probes useful in         │
│  molecular assays; Exscientia automates design of diagnostic reagents and candidate molecules more rapidly;     │
│  BenevolentAI’s knowledge graph and causal-AI uncover mechanistic hypotheses and repurposing opportunities      │
│  that inform diagnostic markers; Recursion combines high‑throughput cellular imaging with machine learning to   │
│  detect disease phenotypes and predictive signatures. Together these advances yield earlier detection, more     │
│  specific biomarkers, and personalized diagnostic pathways that improve clinical decision-making.               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯